# 🧠 Entrenamiento de YOLOv8 para Tumores Cerebrales
Este cuaderno de Google Colab está diseñado para entrenar tu propio modelo de inteligencia artificial médico usando YOLOv8.

**Paso Cero:** Asegúrate de ir al menú superior `Entorno de ejecución > Cambiar tipo de entorno de ejecución` y selecciona **GPU (T4 o superior)**.

## Paso 1: Instalar Dependencias
Instalamos la librería oficial de Ultralytics (YOLO) y Roboflow (para descargar el dataset automáticamente).

In [ ]:
!pip install ultralytics roboflow opencv-python-headless pyyaml

## Paso 2: Importar librerías y validar GPU
Verificaremos que Colab nos ha asignado una GPU de Nvidia correctamente, vital para que el entrenamiento no tarde meses.

In [ ]:
import ultralytics
ultralytics.checks()
import torch
print("GPU Disponible:", torch.cuda.is_available())

## Paso 3: Descargar el Dataset desde Roboflow Universe
**IMPORTANTE**: Debes ir a [universe.roboflow.com](https://universe.roboflow.com/), buscar un dataset de tumores cerebrales (Brain Tumor) en formato YOLOv8, darle a *Download*, elegir *Jupyter/Python* y pegar aquí el código que te genere Roboflow.
Debe verse similar a esto:

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="NV7LwWfEr0Hs1sdAh88q")
project = rf.workspace("roboflow-100").project("brain-tumor-m2pbp")
version = project.version(2)
dataset = version.download("yolov8")

## Paso 3.5: Modificar las etiquetas (Labels) del Dataset
Aquí interceptamos el archivo `data.yaml` que descargó Roboflow y cambiamos los nombres genéricos por los nombres médicos reales antes de que YOLO empiece a entrenar.

In [ ]:
import yaml

# 1. Usamos dataset.location para encontrar dónde guardó Roboflow los datos
ruta_yaml = f"{dataset.location}/data.yaml"

# 2. Abrir y leer el archivo original
with open(ruta_yaml, 'r') as archivo:
    config_dataset = yaml.safe_load(archivo)

# 3. Reemplazar los nombres genéricos (Ajusta los nombres según tus necesidades exactas)
config_dataset['names'] = {
    0: 'Glioma', 
    1: 'Meningioma', 
    2: 'Tumor Pituitario'
}

# 4. Sobreescribir el archivo yaml con los nuevos datos
with open(ruta_yaml, 'w') as archivo:
    yaml.dump(config_dataset, archivo, default_flow_style=False)

print("✅ Nombres de clases actualizados con éxito a:", config_dataset['names'])

## Paso 4: Entrenamiento (Fine-Tuning)
Tomaremos el modelo `yolov8n.pt` base y lo entrenaremos con nuestro dataset médico.
* `data`: Ruta generada por roboflow
* `epochs`: Cuántas veces el modelo repasará las imágenes (sugerimos empezar en 50)
* `imgsz`: Resolución de las imágenes (generalmente 640)
* `patience`: Detener temprano si no hay mejora.

In [ ]:
from ultralytics import YOLO

# 1. Cargar el modelo base nano
model = YOLO('yolov8n.pt')

# 2. Iniciar el entrenamiento
# El atributo dataset.location contiene la ruta donde Roboflow guardó los datos
results = model.train(
    data=f"{dataset.location}/data.yaml", 
    epochs=50, 
    imgsz=640, 
    plots=True, # Generar gráficas de desempeño
    patience=20
)

## Paso 5: Validar los resultados visualmente
Veamos un collage de las predicciones que hizo nuestro nuevo modelo sobre imágenes que nunca había visto (el set de validación).

In [ ]:
from IPython.display import Image, display
import glob

# Mostramos los resultados del entrenamiento
val_batch_images = glob.glob('runs/detect/train/val_batch0_pred.jpg')
if val_batch_images:
    display(Image(filename=val_batch_images[0], width=800))
else:
    print("No se encontraron imágenes de validación. Verifica las rutas en runs/detect/train/")

## Paso 6: Descargar el modelo Médico Final
El resultado final se encuentra en `runs/detect/train/weights/best.pt`. Lo renombraremos para descargarlo y poder usarlo en nuestra API (FastAPI / Dokploy).

In [ ]:
import shutil
from google.colab import files
import os

ruta_original = "runs/detect/train/weights/best.pt"
ruta_nueva = "yolo_tomografia.pt"

if os.path.exists(ruta_original):
    # Copiar y renombrar
    shutil.copy(ruta_original, ruta_nueva)
    # Descargar a tu computadora
    files.download(ruta_nueva)
else:
    print(f"❌ No se encontró el archivo en {ruta_original}. Verifica si el entrenamiento terminó correctamente.")